In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

BASE  = os.path.normpath(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
RAW   = os.path.join(BASE, "data", "raw") + os.sep
PROC  = os.path.join(BASE, "data", "processed") + os.sep
PLOTS = os.path.join(BASE, "plots") + os.sep
OUT   = os.path.join(BASE, "outputs") + os.sep

for d in [PROC, PLOTS, OUT]:
    os.makedirs(d, exist_ok=True)

print("Libraries loaded")
print("BASE:", BASE)

Libraries loaded
BASE: c:\Users\hp\OneDrive\Desktop\evvvv


In [4]:
acn_raw = pd.read_excel(RAW + "acndata_sessions.json.xlsx", sheet_name="Sheet1")
print("ACN Raw shape:", acn_raw.shape)
print("Columns:", acn_raw.columns.tolist())

ACN Raw shape: (16304, 27)
Columns: ['_meta', 'end', 'min_kWh', 'site', 'start', '_items', '_id', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID', 'timezone', 'userID', 'userInputs', 'WhPerMile', 'kWhRequested', 'milesRequested', 'minutesAvailable', 'modifiedAt', 'paymentRequired', 'requestedDeparture', 'userID.1']


In [5]:
acn = acn_raw[[
    '_id', 'siteID', 'stationID', 'spaceID', 'clusterID',
    'connectionTime', 'disconnectTime', 'doneChargingTime',
    'kWhDelivered', 'userID', 'timezone'
]].copy()

acn.columns = [
    'session_id', 'site_id', 'station_id', 'space_id', 'cluster_id',
    'connection_time', 'disconnect_time', 'done_charging_time',
    'kwh_delivered', 'user_id', 'timezone'
]

for col in ['connection_time', 'disconnect_time', 'done_charging_time']:
    acn[col] = pd.to_datetime(acn[col], utc=True, errors='coerce')

acn = acn.dropna(subset=['connection_time', 'disconnect_time'])
acn = acn[acn['kwh_delivered'] > 0].copy()

acn['session_duration_hr']  = (acn['disconnect_time'] - acn['connection_time']).dt.total_seconds() / 3600
acn['idle_time_hr']         = (acn['disconnect_time'] - acn['done_charging_time']).dt.total_seconds() / 3600
acn['avg_power_kw']         = acn['kwh_delivered'] / acn['session_duration_hr'].replace(0, np.nan)
acn['hour']                 = acn['connection_time'].dt.hour
acn['day_of_week']          = acn['connection_time'].dt.dayofweek
acn['is_weekend']           = acn['day_of_week'].isin([5, 6]).astype(int)
acn['month']                = acn['connection_time'].dt.month
acn['revenue_fixed']        = acn['kwh_delivered'] * 15.0
acn = acn[(acn['session_duration_hr'] > 0) & (acn['session_duration_hr'] < 24)]

print("Cleaned ACN shape:", acn.shape)
print("Date range:", str(acn['connection_time'].min().date()), "to", str(acn['connection_time'].max().date()))
print("Total kWh delivered:", round(acn['kwh_delivered'].sum(), 0))
print("Unique stations:", acn['station_id'].nunique())
print("Unique users:", acn['user_id'].nunique())
acn[['station_id','kwh_delivered','session_duration_hr','idle_time_hr','revenue_fixed']].describe().round(2)

Cleaned ACN shape: (14848, 19)
Date range: 2018-04-25 to 2018-12-16
Total kWh delivered: 132782.0
Unique stations: 54
Unique users: 203


,kwh_delivered,session_duration_hr,idle_time_hr,revenue_fixed
count,14848.00,14848.00,14840.00,14848.00
mean,8.94,5.50,2.34,134.14
std,6.95,3.97,3.20,104.31
min,0.50,0.09,-0.02,7.52
25%,3.99,2.01,0.00,59.79
50%,7.42,4.70,0.71,111.26
75%,13.20,8.67,4.06,197.97
max,69.37,23.99,23.01,1040.60


In [6]:
occ_raw   = pd.read_csv(RAW + "occupancy.csv")
vol_raw   = pd.read_csv(RAW + "volume.csv")
dur_raw   = pd.read_csv(RAW + "duration.csv")
price_raw = pd.read_csv(RAW + "price.csv")
time_df   = pd.read_csv(RAW + "time.csv")
info_df   = pd.read_csv(RAW + "information.csv")

print("Occupancy :", occ_raw.shape)
print("Volume    :", vol_raw.shape)
print("Duration  :", dur_raw.shape)
print("Price     :", price_raw.shape)
print("Time      :", time_df.shape)
print("Info      :", info_df.shape)

Occupancy : (8640, 248)
Volume    : (8640, 248)
Duration  : (8640, 248)
Price     : (8640, 248)
Time      : (8640, 6)
Info      : (247, 10)


In [7]:
time_df['datetime'] = pd.to_datetime(time_df[['year','month','day','hour','minute','second']])
datetime_index = time_df['datetime']

def attach_datetime(df, dt_index):
    df = df.copy()
    df.insert(0, 'datetime', dt_index.values)
    df = df.drop(columns=['timestamp'], errors='ignore')
    df = df.set_index('datetime')
    df.columns = df.columns.astype(str)
    return df

occ   = attach_datetime(occ_raw,   datetime_index)
vol   = attach_datetime(vol_raw,   datetime_index)
dur   = attach_datetime(dur_raw,   datetime_index)
price = attach_datetime(price_raw, datetime_index)

missing_pct   = occ.isnull().mean() * 100
good_stations = missing_pct[missing_pct <= 10].index.tolist()

occ   = occ[good_stations].ffill().bfill()
vol   = vol[[c for c in good_stations if c in vol.columns]].ffill().bfill()
dur   = dur[[c for c in good_stations if c in dur.columns]].ffill().bfill()
price = price[[c for c in good_stations if c in price.columns]].ffill().bfill()

print("Time range:", str(datetime_index.iloc[0]), "to", str(datetime_index.iloc[-1]))
print("Stations retained:", len(good_stations))
print("Remaining NaNs:", occ.isnull().sum().sum())

Time range: 2022-06-19 00:00:00 to 2022-07-18 23:55:00
Stations retained: 247
Remaining NaNs: 0


In [8]:
info_df['grid'] = info_df['grid'].astype(str)
station_meta = info_df[['grid','count','fast_count','slow_count','area','lon','la','CBD','dynamic_pricing']].copy()
station_meta = station_meta.rename(columns={'grid':'station_id','la':'lat'})
station_meta = station_meta.set_index('station_id')

util_df = occ.copy()
for col in util_df.columns:
    if col in station_meta.index:
        total = station_meta.loc[col, 'count']
        if total > 0:
            util_df[col] = (util_df[col] / total).clip(0, 1)

congestion_df = (util_df > 0.8).astype(int)
offpeak_df    = (util_df < 0.3).astype(int)
revenue_df    = vol * price * (5 / 60)

print("Utilization Rate = Occupied Chargers / Total Chargers")
print("Mean utilization:", round(util_df.mean().mean(), 4))
print("Congestion percent:", round(congestion_df.mean().mean()*100, 2))
print("Off-peak percent:", round(offpeak_df.mean().mean()*100, 2))

Utilization Rate = Occupied Chargers / Total Chargers
Mean utilization: 0.2802
Congestion percent: 1.02
Off-peak percent: 60.85


In [9]:
# Show utilization trend for first station as preview
sample_col = util_df.columns[0]
sample_data = util_df[sample_col].iloc[:288]  # first 24 hours

plt.figure(figsize=(12, 3))
plt.plot(sample_data.values, color='steelblue', lw=1.2)
plt.axhline(0.8, color='red', ls='--', lw=1, label='Congestion 80%')
plt.axhline(0.3, color='green', ls='--', lw=1, label='Off-peak 30%')
plt.title('Sample Station Utilization - First 24 Hours')
plt.ylabel('Utilization Rate')
plt.xlabel('5-min Intervals')
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS + "sample_utilization_preview.png", bbox_inches='tight')
plt.show()
print("Station:", sample_col)

Station: 102
